# GC-MGT on LargeST-GBA / PEMS-BAY (Kaggle)

Re-implementation of the Granger causality-guided multi-graph transformer (Wang et al., Discover AI 2026), trained with the
**same hyperparameters as HGP** so that the two models can be compared directly: AdamW, learning rate 3e-4, weight decay 1e-4,
batch 16, up to 40 epochs, 1 warm-up epoch then cosine decay, early stopping with patience 6 on the validation mean MAE,
gradient clipping 5, Huber loss, seeds 42 / 7 / 123, scored at 15 / 30 / 60 minutes. These are fixed in the code
(`gcmgt/config.py`), not in this notebook.

The notebook estimates the fixed Granger prior once (cached), optionally times a short run, then trains one run per seed and
prints a **validation** table. Test metrics stay hidden until `SHOW_TEST = True`.

**Before running**
1. Session options: Accelerator = GPU (T4 or P100, 16 GB). Internet = On if the code is cloned from GitHub.
2. Code: either push the repository and set `REPO_URL`, or add the `GCMGT` folder as a Kaggle dataset (found automatically).
3. Data, one of:
   - `DATASET = "gba"`: add the **prepared** `gba_2021.npz` that the HGP runs used (same sensors, filling and splits, so the
     comparison is on identical windows), or the raw `gba_his_raw_2021.h5` + `gba_rn_adj.npy`.
   - `DATASET = "pems_bay"`: add `pems-bay.h5` and `adj_mx_bay.pkl`.
4. Run the timing test first (`TIMING_TEST = True`); training time per sample on a GPU is not known in advance.

In [ ]:
# ---- settings
DATASET = "gba"            # "gba" (2,056 sensors, flow) or "pems_bay" (325 sensors, speed)
SEEDS = (42,)              # final comparison: (42, 7, 123)
HORIZONS = (3, 6, 12)      # scored horizons. Final runs: (1, 3, 6, 12) adds 5 min; "mean" stays the mean over 15 / 30 / 60 min.
SHOW_TEST = False          # keep False while checking; True only for the final runs
TIMING_TEST = True         # train 1 epoch on 64 windows and print the estimated full-run time
STOP_AFTER_TIMING = True   # stop after the timing test; set False (or TIMING_TEST = False) to run the real training

# training budget. None = the HGP protocol (all windows). Lower EPOCH_WINDOWS only if the timing test says the full run does not fit.
EPOCH_WINDOWS = None       # random training windows per epoch (None = all)
GRANGER_WINDOWS = 8        # day-long windows combined into the fixed Granger prior

CODE_PATH = None           # a local folder holding gcmgt/ (testing); normally None
REPO_URL = "https://github.com/prof-sanjay/GCMGT.git"
RUN_TESTS = True

## 1. Environment

In [ ]:
import glob
import json
import os
import shutil
import subprocess
import sys

import torch

print(sys.version.split()[0], "| torch", torch.__version__, "| cuda", torch.cuda.is_available(),
      torch.cuda.get_device_name(0) if torch.cuda.is_available() else "")
if not torch.cuda.is_available():
    print("warning: no GPU; only PEMS-BAY with a short budget is practical on CPU")

## 2. Code and data

In [ ]:
WORK = os.environ.get("GCMGT_WORK", "/kaggle/working")        # saved output: run folders and the Granger prior
SCRATCH = os.environ.get("GCMGT_SCRATCH", "/tmp/gcmgt")        # code, prepared data and logs: not saved
INPUT = os.environ.get("GCMGT_INPUT", "/kaggle/input")
os.makedirs(SCRATCH, exist_ok=True)
os.makedirs(WORK, exist_ok=True)

if CODE_PATH:
    CODE = CODE_PATH
else:
    hits = glob.glob(f"{INPUT}/**/gcmgt/config.py", recursive=True)
    if hits:
        CODE = os.path.dirname(os.path.dirname(hits[0]))
    elif REPO_URL:
        CODE = f"{SCRATCH}/gcmgt_code"
        shutil.rmtree(CODE, ignore_errors=True)
        subprocess.run(["git", "clone", "--depth", "1", REPO_URL, CODE], check=True)
        found = glob.glob(f"{CODE}/**/gcmgt/config.py", recursive=True)
        assert found, "the cloned repository has no gcmgt/config.py"
        CODE = os.path.dirname(os.path.dirname(found[0]))
    else:
        raise SystemExit("GC-MGT code not found: add the repository as a Kaggle dataset or set REPO_URL")
print("code:", CODE)

files = [p for p in glob.glob(f"{INPUT}/**/*", recursive=True) if os.path.isfile(p)]
base = lambda p: os.path.basename(p).lower()
PREPARED = None
if DATASET == "gba":
    prepared = [p for p in files if base(p).endswith(".npz") and "gba" in base(p)]
    traffic = [p for p in files if base(p).endswith(".h5") and "gba" in base(p)]
    adjacency = [p for p in files if base(p).endswith(".npy") and "adj" in base(p)]
    if prepared:
        PREPARED = prepared[0]
        print("prepared data:", PREPARED, "(the file the HGP runs used)")
    else:
        assert traffic and adjacency, "need gba_2021.npz, or gba_his_raw_2021.h5 and gba_rn_adj.npy, under the Kaggle input folder"
        TRAFFIC, ADJACENCY = traffic[0], adjacency[0]
        print("traffic:", TRAFFIC)
        print("adjacency:", ADJACENCY)
        print("warning: raw files keep all 2,352 sensors; HGP used the 2,056 left after the duplicate filter")
elif DATASET == "pems_bay":
    traffic = [p for p in files if base(p).endswith(".h5") and "pems" in base(p) and "meta" not in base(p)]
    adjacency = [p for p in files if base(p).endswith(".pkl") and "adj" in base(p)]
    assert traffic and adjacency, "no pems-bay.h5 / adj_mx_bay.pkl under the Kaggle input folder"
    TRAFFIC, ADJACENCY = traffic[0], adjacency[0]
    print("traffic:", TRAFFIC)
    print("adjacency:", ADJACENCY)
else:
    raise SystemExit('DATASET must be "gba" or "pems_bay"')

In [ ]:
def run(cmd, keep=None):
    """Run a command in the code folder; print only the lines for which keep(line) is true. The full log goes to a file."""
    name = os.path.basename(cmd[1]) if cmd[1] != "-m" else cmd[2]
    log = open(f"{SCRATCH}/log_{name}.txt", "a")
    with subprocess.Popen(cmd, cwd=CODE, env=dict(os.environ, PYTHONUNBUFFERED="1"), stdout=subprocess.PIPE,
                          stderr=subprocess.STDOUT, text=True) as proc:
        for line in proc.stdout:
            log.write(line)
            if keep is None or keep(line):
                print(line, end="")
    log.close()
    assert proc.returncode == 0, f"{name} exited with {proc.returncode}; see {SCRATCH}/log_{name}.txt"


def last_line(line):
    return " passed" in line or " failed" in line or "error" in line.lower()


def prior_lines(line):
    return line.startswith(("estimating", "  granger window", "Traceback")) or "edges" in line


def train_lines(line):
    s = line.strip()
    if s.startswith("epoch"):
        return int(s.split()[1]) % 5 == 0 or int(s.split()[1]) == 1
    return SHOW_TEST and s.startswith("test")


if RUN_TESTS:
    run([sys.executable, "-m", "pytest", "tests", "-q", "-p", "no:cacheprovider", "-W", "ignore"], keep=last_line)

## 3. Prepare the data
Skipped when a prepared file was found. Otherwise the same `scripts/prepare_data.py` as HGP writes it.

In [ ]:
if PREPARED:
    DATA = PREPARED
else:
    DATA = f"{SCRATCH}/data/{DATASET}.npz"
    if not os.path.exists(DATA):
        run([sys.executable, "scripts/prepare_data.py", "--dataset", DATASET, "--traffic", TRAFFIC, "--adjacency", ADJACENCY, "--out", DATA])
print("data:", DATA)

## 4. Fixed Granger prior
Pairwise Granger F-tests with Benjamini-Hochberg control inside each day-long window, estimated once from the training split and cached in `graphs/`. The regressions run on the CPU (about 2 minutes per window at 2,056 sensors on a laptop), so expect roughly 20 minutes for the default 8 windows; a re-run reuses the cache.

In [ ]:
GRAPHS = f"{WORK}/graphs"
run([sys.executable, "scripts/build_prior.py", "--data", DATA, "--graphs", GRAPHS, "--granger-windows", str(GRANGER_WINDOWS)], keep=prior_lines)

## 5. Timing test (optional)
Trains one epoch on 64 windows and extrapolates to the full protocol. Use it to decide whether 40 epochs per seed fit in the session, and what `EPOCH_WINDOWS` to use otherwise.

In [ ]:
if TIMING_TEST:
    sys.path.insert(0, CODE)
    from gcmgt.config import DataConfig
    from gcmgt.data import Windows, load

    dc = DataConfig.from_file(DATA)
    data = load(dc)
    steps = tuple(range(1, dc.out_steps + 1))
    n_train, n_val = (len(Windows(s, dc.seq_len, steps)) for s in (data.train, data.val))
    probe = f"{SCRATCH}/timing"
    shutil.rmtree(probe, ignore_errors=True)
    run([sys.executable, "scripts/train.py", "--data", DATA, "--graphs", GRAPHS, "--results", probe, "--granger-windows", str(GRANGER_WINDOWS),
         "--epochs", "1", "--max-windows", "64"], keep=lambda l: l.strip().startswith("epoch"))
    sec = float(open(f"{probe}/gcmgt__seed42/history.csv").read().splitlines()[1].split(",")[-1])
    per_window = sec / 128                                    # 64 training + 64 validation windows (validation is cheaper: conservative)
    epoch = per_window * (min(EPOCH_WINDOWS or n_train, n_train) + n_val)
    print(f"windows: train {n_train:,}  validation {n_val:,}")
    print(f"estimated {epoch / 60:.1f} min per epoch, {40 * epoch / 3600:.1f} h for 40 epochs per seed (early stopping may end sooner)")
    print("A Kaggle session lasts at most 9-12 h: if this is too long, set EPOCH_WINDOWS to a smaller number (and report it).")
    del data
    if STOP_AFTER_TIMING:
        print("timing test done; set STOP_AFTER_TIMING = False (or TIMING_TEST = False) and run again to train")

## 6. Train and evaluate
One run per seed. Each writes `results.json`, `history.csv`, `config.json`, `checkpoints/best.pt` and `test_predictions_h<h>.npz` into `results/gcmgt__seed<seed>/` (the same layout as the HGP runs, so paired tests work across the two models).

In [ ]:
RESULTS = f"{WORK}/results"
if TIMING_TEST and STOP_AFTER_TIMING:
    print("skipped: STOP_AFTER_TIMING is True")
else:
    for seed in SEEDS:
        if os.path.exists(f"{RESULTS}/gcmgt__seed{seed}/results.json"):
            print("already done: seed", seed)
            continue
        print("--- seed", seed)
        cmd = [sys.executable, "scripts/train.py", "--seed", str(seed), "--data", DATA, "--graphs", GRAPHS, "--results", RESULTS,
               "--granger-windows", str(GRANGER_WINDOWS), "--horizons", *map(str, HORIZONS)]
        if EPOCH_WINDOWS:
            cmd += ["--epoch-windows", str(EPOCH_WINDOWS)]
        run(cmd, keep=train_lines)
        r = json.load(open(f"{RESULTS}/gcmgt__seed{seed}/results.json"))
        print(f"done: best epoch {r['best_epoch']} of {r['epochs']} run (max 40), best val MAE {r['validation']['mean']['mae']:.3f}, "
              f"{r['training_seconds'] / 60:.1f} min")

## 7. Validation table
Every run under `results/`. The recipe actually used is printed from `config.json` so it can be checked against HGP.

In [ ]:
rows = []
for f in sorted(glob.glob(f"{RESULTS}/*/results.json")):
    r = json.load(open(f))
    v = r["validation"]
    rows.append((r["run"], v["mean"]["mae"], v["3"]["mae"], v["6"]["mae"], v["12"]["mae"], v["mean"]["rmse"], r["best_epoch"], r["training_seconds"] / 60))
print(f"{'run':16s} {'valMAE':>7s} {'15min':>6s} {'30min':>6s} {'60min':>6s} {'valRMSE':>8s} {'best':>4s} {'min':>6s}")
for name, mae, a, b, c, rmse, ep, minutes in rows:
    print(f"{name:16s} {mae:7.3f} {a:6.3f} {b:6.3f} {c:6.3f} {rmse:8.3f} {ep:4d} {minutes:6.1f}")

cfgs = sorted(glob.glob(f"{RESULTS}/*/config.json"))
if cfgs:
    t = json.load(open(cfgs[0]))["train"]
    keys = ("optimizer", "learning_rate", "weight_decay", "batch_size", "micro_batch", "epochs", "schedule", "warmup_epochs",
            "patience", "grad_clip", "loss", "huber_delta", "epoch_windows")
    print("\nrecipe used:", {k: t[k] for k in keys})

if SHOW_TEST:
    for f in sorted(glob.glob(f"{RESULTS}/*/results.json")):
        r = json.load(open(f))
        print(r["run"], "test:", "  ".join(f"h{h}: RMSE {r['test'][str(h)]['rmse']:.3f} MAE {r['test'][str(h)]['mae']:.3f}" for h in HORIZONS))